In [1]:
# Standard libraries
import os
import sys
import json
from pathlib import Path

# Data analysis
import pandas as pd

# Notebook display
from IPython.display import display

# PySpark
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [2]:
PROJECT_ROOT = Path(
    r"D:\projects\AI_Monitoring_Project"
)

ADVANCED_DIR = PROJECT_ROOT / "advanced"
ETL_DIR = ADVANCED_DIR / "ETL"

WAREHOUSE_DIR = ETL_DIR / "warehouse"

BRONZE_WAREHOUSE_DIR = WAREHOUSE_DIR / "bronze"
SILVER_WAREHOUSE_DIR = WAREHOUSE_DIR / "silver"
GOLD_WAREHOUSE_DIR = WAREHOUSE_DIR / "gold"

SPARK_TEMP_DIR = ETL_DIR / ".spark-temp"

METASTORE_DB_DIR = ETL_DIR / "metastore_db"

GOLD_WAREHOUSE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPARK_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project paths configured successfully.")

Project paths configured successfully.


In [3]:
JAVA_HOME = Path(
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

HADOOP_HOME = Path(
    r"C:\hadoop\hadoop-3.5.0"
)

required_files = [
    JAVA_HOME / "bin" / "java.exe",
    HADOOP_HOME / "bin" / "winutils.exe",
    HADOOP_HOME / "bin" / "hadoop.dll",
    HADOOP_HOME / "bin" / "hdfs.dll"
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(file_path)

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)

os.environ["PATH"] = os.pathsep.join([
    str(JAVA_HOME / "bin"),
    str(HADOOP_HOME / "bin"),
    os.environ.get("PATH", "")
])

print("Java and Hadoop configured successfully.")

Java and Hadoop configured successfully.


In [4]:
# Configure Derby Metastore
METASTORE_URL = (
    f"jdbc:derby:;databaseName="
    f"{METASTORE_DB_DIR.as_posix()};create=true"
)

# Initialize SparkSession
spark = (
    SparkSession.builder
    .appName("AI_Monitoring_Gold")
    .master("local[2]")
    .config(
        "spark.sql.warehouse.dir",
        WAREHOUSE_DIR.as_uri()
    )
    .config(
        "spark.local.dir",
        str(SPARK_TEMP_DIR)
    )
    .config(
        "spark.sql.catalogImplementation",
        "hive"
    )
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        METASTORE_URL
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession initialized successfully!")

SparkSession initialized successfully!


In [5]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS gold
    LOCATION '{GOLD_WAREHOUSE_DIR.as_uri()}'
""")

print("Gold database is ready.")

Gold database is ready.


In [7]:
silver_tables = [
    table.name
    for table in spark.catalog.listTables("silver")
]

print("Available Silver tables:")

for table in silver_tables:
    print("-", table)

Available Silver tables:
- cameras
- employees
- person_detection_details
- person_detections
- plate_detections
- restricted_access


In [8]:
silver_dataframes = {
    table_name: spark.table(f"silver.{table_name}")
    for table_name in silver_tables
}

print("Successfully loaded Silver DataFrames:")

Successfully loaded Silver DataFrames:


In [9]:
person_detections_df = silver_dataframes["person_detections"]

person_detection_details_df = silver_dataframes["person_detection_details"]

plate_detections_df = silver_dataframes["plate_detections"]

employees_df = silver_dataframes["employees"]

cameras_df = silver_dataframes["cameras"]

restricted_access_df = silver_dataframes["restricted_access"]

print("All six DataFrames are ready.")

All six DataFrames are ready.


In [11]:
display(person_detections_df.toPandas())

,Time_stamp,Zone,detected_persons_id,authorized,camera_id,event_id,detector_type
0,2026-10-28 12:09:58.634651,B,"[EMP0031, EMP0070, EMP0087]","[True, True, True]",004,camera.person.frames.dev:1:3480,synthetic
1,2026-10-28 12:09:58.634651,C,[],[],005,camera.person.frames.dev:1:3481,synthetic
2,2026-10-28 12:09:58.634651,D,[],[],006,camera.person.frames.dev:1:3482,synthetic
3,2026-10-28 12:09:58.634651,D,[],[],007,camera.person.frames.dev:1:3483,synthetic
4,2026-10-28 12:09:59.693337,B,"[EMP0031, EMP0070, EMP0087]","[True, True, True]",004,camera.person.frames.dev:1:3484,synthetic
...,...,...,...,...,...,...,...
39251,2026-10-28 15:51:41.090889,B,"[EMP0020, EMP0029, EMP0087]","[True, True, True]",003,camera.person.frames.dev:0:19460,synthetic
39252,2026-10-28 15:51:41.090889,B,"[EMP0017, EMP0021, EMP0072]","[True, True, True]",004,camera.person.frames.dev:1:25908,synthetic
39253,2026-10-28 15:51:41.094153,C,[],[],005,camera.person.frames.dev:1:25909,synthetic
39254,2026-10-28 15:51:41.094657,D,[],[],006,camera.person.frames.dev:1:25910,synthetic


In [12]:
display(person_detection_details_df.toPandas())

,Time_stamp,Zone,camera_id,event_id,detector_type,employee_id,authorized
0,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,EMP0029,True
1,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,EMP0072,True
2,2026-10-28 12:09:58.634651,A,002,camera.person.frames.dev:0:2638,synthetic,EMP0088,True
3,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,EMP0029,True
4,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,EMP0072,True
...,...,...,...,...,...,...,...
36468,2026-10-28 15:51:34.016769,B,004,camera.person.frames.dev:1:25880,synthetic,EMP0021,True
36469,2026-10-28 15:51:34.016769,B,004,camera.person.frames.dev:1:25880,synthetic,EMP0072,True
36470,2026-10-28 15:51:35.029525,B,004,camera.person.frames.dev:1:25884,synthetic,EMP0017,True
36471,2026-10-28 15:51:35.029525,B,004,camera.person.frames.dev:1:25884,synthetic,EMP0021,True


In [13]:
display(plate_detections_df.toPandas())

,Time_stamp,Zone,detected_plates,camera_id,event_id,detector_type
0,2026-10-28 12:00:27.299987,E,[],008,camera.plate.frames.dev:0:1821,synthetic
1,2026-10-28 12:00:27.299987,E,[],009,camera.plate.frames.dev:0:1822,synthetic
2,2026-10-28 12:00:27.299987,E,[],010,camera.plate.frames.dev:0:1823,synthetic
3,2026-10-28 12:00:28.351386,E,[],008,camera.plate.frames.dev:0:1824,synthetic
4,2026-10-28 12:00:28.354365,E,[],009,camera.plate.frames.dev:0:1825,synthetic
...,...,...,...,...,...,...
17608,2026-10-28 15:51:40.086463,E,[],009,camera.plate.frames.dev:0:19429,synthetic
17609,2026-10-28 15:51:40.087033,E,[],010,camera.plate.frames.dev:0:19430,synthetic
17610,2026-10-28 15:51:41.095783,E,[],008,camera.plate.frames.dev:0:19431,synthetic
17611,2026-10-28 15:51:41.096319,E,[],009,camera.plate.frames.dev:0:19432,synthetic


In [14]:
# Define array schema
plate_array_schema = T.ArrayType(
    T.StringType()
)

# Parse JSON strings into arrays
plate_detections_parsed_df = (
    plate_detections_df
    .withColumn(
        "detected_plates",
        F.from_json(
            F.col("detected_plates"),
            plate_array_schema
        )
    )
)

In [15]:
employee_plate_lookup_df = (
    employees_df
    .filter(
        F.col("plate_number").isNotNull() &
        (F.trim(F.col("plate_number")) != "")
    )
    .select(
        F.upper(
            F.trim(F.col("plate_number"))
        ).alias("_plate_key"),

        F.col("employee_id")
    )
)

In [16]:
employee_plate_map_df = (
    employee_plate_lookup_df
    .agg(
        F.map_from_entries(
            F.collect_list(
                F.struct(
                    F.col("_plate_key"),
                    F.col("employee_id")
                )
            )
        ).alias("_plate_to_employee")
    )
)

In [23]:
plate_events_df = (
    plate_detections_parsed_df

    # Attach employee lookup map
    .crossJoin(employee_plate_map_df)

    # Map each detected plate to its employee ID
    .withColumn(
        "detected_employee_ids",
        F.transform(
            F.col("detected_plates"),
            lambda plate: F.element_at(
                F.col("_plate_to_employee"),
                F.upper(F.trim(plate))
            )
        )
    )

    # Remove temporary lookup map
    .drop("_plate_to_employee")

    # Identify detection source
    .withColumn(
        "detection_source",
        F.lit("plate")
    )
)

In [24]:
display(
    plate_events_df
    .select(
        "Time_stamp",
        "Zone",
        "detected_plates",
        "detected_employee_ids",
        "camera_id",
        "event_id"
    )
    .limit(500)
    .toPandas()
)

,Time_stamp,Zone,detected_plates,detected_employee_ids,camera_id,event_id
0,2026-10-28 12:00:27.299987,E,[],[],008,camera.plate.frames.dev:0:1821
1,2026-10-28 12:00:27.299987,E,[],[],009,camera.plate.frames.dev:0:1822
2,2026-10-28 12:00:27.299987,E,[],[],010,camera.plate.frames.dev:0:1823
3,2026-10-28 12:00:28.351386,E,[],[],008,camera.plate.frames.dev:0:1824
4,2026-10-28 12:00:28.354365,E,[],[],009,camera.plate.frames.dev:0:1825
...,...,...,...,...,...,...
495,2026-10-28 12:03:14.361142,E,[],[],008,camera.plate.frames.dev:0:2316
496,2026-10-28 12:03:14.362193,E,[],[],009,camera.plate.frames.dev:0:2317
497,2026-10-28 12:03:14.363284,E,[],[],010,camera.plate.frames.dev:0:2318
498,2026-10-28 12:03:15.379668,E,[],[],008,camera.plate.frames.dev:0:2319


In [25]:
plate_details_df = (
    plate_events_df

    # Pair each plate with its corresponding employee ID
    .withColumn(
        "_plate_details",
        F.explode(
            F.arrays_zip(
                F.col("detected_plates"),
                F.col("detected_employee_ids")
            )
        )
    )

    # Select the required columns
    .select(
        "Time_stamp",
        "Zone",
        "camera_id",
        "event_id",
        "detector_type",

        F.col(
            "_plate_details.detected_plates"
        ).alias("plate_number"),

        F.col(
            "_plate_details.detected_employee_ids"
        ).alias("employee_id"),

        F.lit("plate").alias("detection_source")
    )
)

In [26]:
dataframes_to_check = {
    "person_detections_df": person_detections_df,
    "plate_events_df": plate_events_df,
    "person_detection_details_df": person_detection_details_df,
    "plate_details_df": plate_details_df
}

In [27]:

for name, df in dataframes_to_check.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Columns:", df.columns)
    df.printSchema()


person_detections_df
Columns: ['Time_stamp', 'Zone', 'detected_persons_id', 'authorized', 'camera_id', 'event_id', 'detector_type']
root
 |-- Time_stamp: timestamp (nullable = true)
 |-- Zone: string (nullable = true)
 |-- detected_persons_id: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- authorized: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)


plate_events_df
Columns: ['Time_stamp', 'Zone', 'detected_plates', 'camera_id', 'event_id', 'detector_type', 'detected_employee_ids', 'detection_source']
root
 |-- Time_stamp: timestamp (nullable = true)
 |-- Zone: string (nullable = true)
 |-- detected_plates: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)
 |-- 

In [28]:
def normalized_column(
    df,
    candidates,
    data_type,
    required=False
):
    column_name = next(
        (c for c in candidates if c in df.columns),
        None
    )

    if column_name is None:
        if required:
            raise ValueError(
                f"Required column not found. "
                f"Expected one of: {candidates}"
            )

        return F.lit(None).cast(data_type)

    return F.col(column_name).cast(data_type)

In [29]:
person_events_normalized_df = (
    person_detections_df
    .select(
        normalized_column(
            person_detections_df,
            ["Time_stamp", "ts", "timestamp"],
            T.TimestampType(),
            True
        ).alias("timestamp"),

        normalized_column(
            person_detections_df,
            ["Zone", "zone"],
            T.StringType()
        ).alias("Zone"),

        normalized_column(
            person_detections_df,
            ["camera_id"],
            T.StringType()
        ).alias("camera_id"),

        normalized_column(
            person_detections_df,
            ["event_id"],
            T.StringType()
        ).alias("event_id"),

        normalized_column(
            person_detections_df,
            ["detector_type"],
            T.StringType()
        ).alias("detector_type"),

        F.lit("person").alias("detection_source"),

        normalized_column(
            person_detections_df,
            ["detected_persons_id", "detected_employee_ids"],
            T.ArrayType(T.StringType())
        ).alias("detected_employee_ids"),

        F.lit(None).cast(
            T.ArrayType(T.StringType())
        ).alias("detected_plates"),

        normalized_column(
            person_detections_df,
            ["authorized"],
            T.ArrayType(T.BooleanType())
        ).alias("authorized")
    )
)

In [30]:
plate_events_normalized_df = (
    plate_events_df
    .select(
        normalized_column(
            plate_events_df,
            ["Time_stamp", "ts", "timestamp"],
            T.TimestampType(),
            True
        ).alias("timestamp"),

        normalized_column(
            plate_events_df,
            ["Zone", "zone"],
            T.StringType()
        ).alias("Zone"),

        normalized_column(
            plate_events_df,
            ["camera_id"],
            T.StringType()
        ).alias("camera_id"),

        normalized_column(
            plate_events_df,
            ["event_id"],
            T.StringType()
        ).alias("event_id"),

        normalized_column(
            plate_events_df,
            ["detector_type"],
            T.StringType()
        ).alias("detector_type"),

        F.lit("plate").alias("detection_source"),

        normalized_column(
            plate_events_df,
            ["detected_employee_ids"],
            T.ArrayType(T.StringType())
        ).alias("detected_employee_ids"),

        normalized_column(
            plate_events_df,
            ["detected_plates"],
            T.ArrayType(T.StringType())
        ).alias("detected_plates"),

        F.lit(None).cast(
            T.ArrayType(T.BooleanType())
        ).alias("authorized")
    )
)

In [31]:
monitoring_events_df = (
    person_events_normalized_df
    .unionByName(
        plate_events_normalized_df
    )
)

In [32]:
display(
    monitoring_events_df
    .limit(500)
    .toPandas()
)

,timestamp,Zone,camera_id,event_id,detector_type,detection_source,detected_employee_ids,detected_plates,authorized
0,2026-10-28 12:09:58.634651,B,004,camera.person.frames.dev:1:3480,synthetic,person,"[EMP0031, EMP0070, EMP0087]",None,"[True, True, True]"
1,2026-10-28 12:09:58.634651,C,005,camera.person.frames.dev:1:3481,synthetic,person,[],None,[]
2,2026-10-28 12:09:58.634651,D,006,camera.person.frames.dev:1:3482,synthetic,person,[],None,[]
3,2026-10-28 12:09:58.634651,D,007,camera.person.frames.dev:1:3483,synthetic,person,[],None,[]
4,2026-10-28 12:09:59.693337,B,004,camera.person.frames.dev:1:3484,synthetic,person,"[EMP0031, EMP0070, EMP0087]",None,"[True, True, True]"
...,...,...,...,...,...,...,...,...,...
495,2026-10-28 12:12:03.139496,D,007,camera.person.frames.dev:1:3975,synthetic,person,[],None,[]
496,2026-10-28 12:12:04.149781,B,004,camera.person.frames.dev:1:3976,synthetic,person,"[EMP0059, EMP0084]",None,"[True, True]"
497,2026-10-28 12:12:04.150832,C,005,camera.person.frames.dev:1:3977,synthetic,person,[],None,[]
498,2026-10-28 12:12:04.151350,D,006,camera.person.frames.dev:1:3978,synthetic,person,[],None,[]


In [33]:
person_details_normalized_df = (
    person_detection_details_df
    .select(
        normalized_column(
            person_detection_details_df,
            ["Time_stamp", "ts", "timestamp"],
            T.TimestampType(),
            True
        ).alias("timestamp"),

        normalized_column(
            person_detection_details_df,
            ["Zone", "zone"],
            T.StringType()
        ).alias("Zone"),

        normalized_column(
            person_detection_details_df,
            ["camera_id"],
            T.StringType()
        ).alias("camera_id"),

        normalized_column(
            person_detection_details_df,
            ["event_id"],
            T.StringType()
        ).alias("event_id"),

        normalized_column(
            person_detection_details_df,
            ["detector_type"],
            T.StringType()
        ).alias("detector_type"),

        F.lit("person").alias("detection_source"),

        normalized_column(
            person_detection_details_df,
            [
                "employee_id",
                "detected_persons_id",
                "detected_person_id"
            ],
            T.StringType()
        ).alias("employee_id"),

        F.lit(None).cast(
            T.StringType()
        ).alias("plate_number"),

        normalized_column(
            person_detection_details_df,
            ["authorized"],
            T.BooleanType()
        ).alias("authorized")
    )
)

In [34]:
plate_details_normalized_df = (
    plate_details_df
    .select(
        normalized_column(
            plate_details_df,
            ["Time_stamp", "ts", "timestamp"],
            T.TimestampType(),
            True
        ).alias("timestamp"),

        normalized_column(
            plate_details_df,
            ["Zone", "zone"],
            T.StringType()
        ).alias("Zone"),

        normalized_column(
            plate_details_df,
            ["camera_id"],
            T.StringType()
        ).alias("camera_id"),

        normalized_column(
            plate_details_df,
            ["event_id"],
            T.StringType()
        ).alias("event_id"),

        normalized_column(
            plate_details_df,
            ["detector_type"],
            T.StringType()
        ).alias("detector_type"),

        F.lit("plate").alias("detection_source"),

        normalized_column(
            plate_details_df,
            ["employee_id"],
            T.StringType()
        ).alias("employee_id"),

        normalized_column(
            plate_details_df,
            ["plate_number"],
            T.StringType()
        ).alias("plate_number"),

        F.lit(None).cast(
            T.BooleanType()
        ).alias("authorized")
    )
)

In [35]:
monitoring_detection_details_df = (
    person_details_normalized_df
    .unionByName(
        plate_details_normalized_df
    )
)

In [36]:
display(
    monitoring_detection_details_df
    .limit(500)
    .toPandas()
)



,timestamp,Zone,camera_id,event_id,detector_type,detection_source,employee_id,plate_number,authorized
0,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,person,EMP0029,None,True
1,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,person,EMP0072,None,True
2,2026-10-28 12:09:58.634651,A,002,camera.person.frames.dev:0:2638,synthetic,person,EMP0088,None,True
3,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,person,EMP0029,None,True
4,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,person,EMP0072,None,True
...,...,...,...,...,...,...,...,...,...
495,2026-10-28 12:11:40.871799,A,002,camera.person.frames.dev:0:2941,synthetic,person,EMP0077,None,True
496,2026-10-28 12:11:40.872429,B,003,camera.person.frames.dev:0:2942,synthetic,person,EMP0066,None,True
497,2026-10-28 12:11:41.880662,A,001,camera.person.frames.dev:0:2943,synthetic,person,EMP0029,None,True
498,2026-10-28 12:11:41.880662,A,001,camera.person.frames.dev:0:2943,synthetic,person,EMP0072,None,True


In [37]:
employees_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- plate_number: string (nullable = true)



In [38]:
print("Total employees:", employees_df.count())

Total employees: 100


In [43]:
display(
    employees_df
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .toPandas()
)

,employee_id,count


In [44]:
monitoring_detection_details_df.write.mode("overwrite").saveAsTable("gold.monitoring_detection_details")

In [47]:
monitoring_events_df.write.mode("overwrite").saveAsTable("gold.monitoring_events")

In [49]:
employee_directory_df  = (employees_df
    .select("employee_id", "plate_number")
)

In [50]:
display(employee_directory_df.toPandas())

,employee_id,plate_number
0,EMP0001,None
1,EMP0002,SYN-4230
2,EMP0003,SYN-1969
3,EMP0004,SYN-9617
4,EMP0005,None
...,...,...
95,EMP0096,SYN-3648
96,EMP0097,SYN-4724
97,EMP0098,SYN-9403
98,EMP0099,SYN-1165


In [51]:
employee_directory_df.write.mode("overwrite").saveAsTable("gold.employee_directory")

In [52]:
restricted_access_df.write.mode("overwrite").saveAsTable("gold.restricted_access")